In [ ]:
import shutil
shutil.copytree(r"/kaggle/input/datasets/xiaokonglong80/yolo11-hand-gesture-recognition", r"/kaggle/working/yolo11-hand-gesture-recognition")

In [ ]:
!pip install -e /kaggle/working/yolo11-hand-gesture-recognition

In [ ]:
!pip uninstall -y ray

In [ ]:
%cd /kaggle/working/yolo11-hand-gesture-recognition/ultralytics/cfg/datasets

In [ ]:
%%writefile GES.yaml
# Ultralytics YOLO 🚀, AGPL-3.0 license
# COCO128 dataset https://www.kaggle.com/datasets/ultralytics/coco128 (first 128 images from COCO train2017) by Ultralytics
# Documentation: https://docs.ultralytics.com/datasets/detect/coco/
# Example usage: yolo train data=coco128.yaml
# parent
# ├── ultralytics
# └── datasets
#     └── coco128  ← downloads here (7 MB)

# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: /kaggle/working/yolo11-hand-gesture-recognition/ges_dataset # dataset root dir
train: train/images  # train images (relative to 'path')
val: val/images  # val images (relative to 'path')
test: test/images  # test images (relative to 'path') 

# Classes
nc: 7  # number of classes
names:
  0: forward
  1: backward
  2: left
  3: right
  4: rotate_left
  5: rotate_right
  6: stop


In [ ]:
%cd /kaggle/working/yolo11-hand-gesture-recognition

In [ ]:
%%writefile train.py
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# 强制将当前目录加入系统路径，确保 DDP 子进程能找到 ultralytics
sys.path.append(os.getcwd())

from ultralytics import YOLO

if __name__ == '__main__':
    os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"

    os.environ["WANDB_API_KEY"] = "f4006571baebc40f3fe00fe509979ec34ff6455c"
    os.environ["WANDB_MODE"] = "online"
    model = YOLO('yolo11n.yaml') 
    model.train(data=r'GES.yaml',
                cache='ram',
                imgsz=640,
                epochs=500,
                batch=128,          
                device=[0, 1],      
                project='runs/train',
                name='exp',
                cfg='config.yaml',
                workers=4           # DDP 模式下 workers 不要设太高，防止内存溢出
                )

In [ ]:
# !python train.py
!python -m torch.distributed.run --nproc_per_node 2 ./train.py

In [ ]:
!zip -q -r /kaggle/working/train_results.zip /kaggle/working/yolo11-hand-gesture-recognition/runs